# EEG Mental Workload Classifier — Phase 2: Preprocessing, Feature Extraction, Baseline Model

**What this notebook does, in order:**
1. Load the checkpointed arrays from Notebook 1
2. Bandpass filter (4-45Hz) to remove drift and high-frequency noise
3. Run ICA to identify and remove eye-blink artifacts (v2: signal-correlation based)
4. Window each 150-second recording into short epochs (turns 45 recordings into hundreds of training examples)
5. Extract theta/alpha/beta/gamma band-power features per channel per window
6. Sanity-check the features against cognitive load theory using medians (robust to outliers)
7. Train baseline models (Random Forest, SVM) with **subject-independent** cross-validation
8. Explicitly demonstrate the leakage problem by comparing against a naive (non-subject-independent) split

**Read the markdown before each code cell** -- it explains what the cell is doing and, more importantly, what output would indicate the step worked vs. didn't.

## 0. Setup + load checkpoint

In [ ]:
!pip install -q mne scipy scikit-learn matplotlib seaborn

from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/eeg-workload-project'
os.makedirs(f'{PROJECT_DIR}/figures', exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mne
from scipy.signal import welch

sns.set_theme(style='whitegrid')
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
mne.set_log_level('WARNING')

N_CHANNELS = 14
SFREQ = 128
CHANNEL_NAMES = ['AF3','F7','F3','FC5','T7','P7','O1','O2','P8','T8','FC6','F4','F8','AF4']

X = np.load(f'{PROJECT_DIR}/data/processed/X_raw.npy')   # shape (45, 14, 19200)
y = np.load(f'{PROJECT_DIR}/data/processed/y_class012.npy')

print('X shape:', X.shape)
print('y shape:', y.shape)

subject_id = np.arange(X.shape[0])
print('subject_id:', subject_id)

## 1. Bandpass filter

**What this does:** removes everything outside 4-45Hz -- slow drift below 4Hz, muscle/powerline noise above 45Hz -- leaving the theta/alpha/beta/gamma range where workload-related brain activity lives.

**What "working" looks like:** the dominant near-0Hz spike from raw data should be gone in the post-filter PSD, replaced by a flatter curve with visible structure across 4-45Hz. You already confirmed this in your last run -- no changes needed here.

In [ ]:
def bandpass_filter(X, sfreq=SFREQ, l_freq=4.0, h_freq=45.0):
    X_filtered = np.zeros_like(X)
    for i in range(X.shape[0]):
        X_filtered[i] = mne.filter.filter_data(
            X[i].astype(np.float64), sfreq=sfreq, l_freq=l_freq, h_freq=h_freq, verbose=False
        )
    return X_filtered

X_filtered = bandpass_filter(X)
print('Filtered shape:', X_filtered.shape)

In [ ]:
def plot_psd_comparison(X_before, X_after, idx, sfreq=SFREQ, channel_names=CHANNEL_NAMES):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
    for ax, data, title in zip(axes, [X_before, X_after], ['Before filtering', 'After filtering']):
        sample = data[idx]
        for i in range(sample.shape[0]):
            freqs, psd = welch(sample[i], fs=sfreq, nperseg=min(512, sample.shape[1]))
            ax.semilogy(freqs, psd, linewidth=0.7)
        ax.set_xlim(0, 50)
        ax.set_xlabel('Frequency (Hz)')
        ax.set_title(title)
    axes[0].set_ylabel('PSD (log scale)')
    plt.tight_layout()
    plt.savefig(f'{PROJECT_DIR}/figures/psd_before_after_filter.png', dpi=150)
    plt.show()

plot_psd_comparison(X, X_filtered, idx=0)

## 2. ICA-based eye-blink removal (v3 -- top-N rank-based, not threshold-based)

**What this does:** decomposes each recording's 14 channels into 14 statistically independent components. Blinks tend to concentrate almost entirely into one or two components. We correlate each component against a frontal pseudo-EOG proxy channel and remove the most-correlated one(s), then reconstruct the signal.

**Why we changed approach again:** v1 (topography-based) removed nothing effective. v2 (correlation threshold=3.0) was too conservative -- ~half of subjects got zero components removed, still no visible cleaning. Lowering the threshold to 1.5 fixed the under-removal problem, but overshot in the other direction: 3-5 components were now removed per subject, subject-independent model accuracy actually **dropped below chance** (RF: 0.289, SVM: 0.337), and several posterior channels that had no visible blink artifact in the first place looked *more* noisy after cleaning, not less. That combination is the signature of removing real neural signal (likely some genuine frontal theta activity, which is also frontally-weighted) along with the blinks -- an arbitrary correlation z-score threshold can over-fire depending on how correlated a given subject's brain activity happens to be with their own frontal channel.

**The fix:** stop thresholding on a z-score entirely. Instead, always remove exactly the **top-1 most correlated component** per subject (configurable via `N_COMPONENTS_TO_REMOVE`). A genuine blink component is typically dramatically more correlated with a frontal channel than any real brain-activity component, so ranking and taking a small fixed number is safer and more standard than a threshold that can swing between under- and over-removal.

**What "working" looks like:** re-plotting epoch #0, the frontal spikes at ~13s/~37s should be visibly reduced (unlike v2's threshold=3.0 attempt), *without* posterior channels (P7, O1, O2, P8) looking more contaminated afterward than before (unlike v2's threshold=1.5 attempt). If subject-independent model accuracy (Section 6) comes back closer to run 1's numbers (RF ~0.35-0.45, SVM ~0.45-0.5) rather than below chance, that's a strong sign we've found a better balance.

In [ ]:
PSEUDO_EOG_CHANNEL = 'AF3'  # frontal channel used as an eye-movement proxy -- STEW has no dedicated EOG channel
N_COMPONENTS_TO_REMOVE = 1  # fixed count, not a threshold -- see markdown above for why

def build_mne_info():
    return mne.create_info(ch_names=CHANNEL_NAMES, sfreq=SFREQ, ch_types='eeg')

def remove_blink_components(sample, info, pseudo_eog_ch=PSEUDO_EOG_CHANNEL, n_remove=N_COMPONENTS_TO_REMOVE):
    """Run ICA, rank all components by |correlation| with a frontal pseudo-EOG
    proxy channel, and remove exactly the top `n_remove` most-correlated ones
    (rather than everything exceeding an arbitrary threshold, which proved
    unstable -- see markdown above)."""
    raw = mne.io.RawArray(sample, info, verbose=False)
    ica = mne.preprocessing.ICA(n_components=min(14, sample.shape[0]), random_state=RANDOM_SEED, max_iter='auto')
    ica.fit(raw, verbose=False)

    # get_sources gives us each component's time series; correlate each
    # against the raw pseudo-EOG channel's time series directly.
    sources = ica.get_sources(raw).get_data()  # shape (n_components, n_timepoints)
    eog_signal = raw.get_data(picks=[pseudo_eog_ch])[0]
    correlations = np.array([np.abs(np.corrcoef(sources[c], eog_signal)[0, 1]) for c in range(sources.shape[0])])

    blink_components = list(np.argsort(correlations)[-n_remove:])
    ica.exclude = blink_components
    raw_clean = raw.copy()
    ica.apply(raw_clean, verbose=False)
    return raw_clean.get_data(), blink_components, correlations[blink_components]

info = build_mne_info()
X_clean = np.zeros_like(X_filtered)
excluded_log = []
corr_log = []

for i in range(X_filtered.shape[0]):
    cleaned, excluded, corrs = remove_blink_components(X_filtered[i], info)
    X_clean[i] = cleaned
    excluded_log.append(excluded)
    corr_log.append(corrs)
    print(f'Sample {i}: removed component(s) {excluded}, correlation(s) with {PSEUDO_EOG_CHANNEL} = {np.round(corrs, 3)}')

print('\nDone. X_clean shape:', X_clean.shape)
print('Mean correlation of removed component(s) across all subjects:', np.mean([c for corrs in corr_log for c in corrs]).round(3))

In [ ]:
def plot_before_after_ica(X_before, X_after, idx, sfreq=SFREQ, channel_names=CHANNEL_NAMES, seconds=40):
    n_points = int(seconds * sfreq)
    t = np.arange(n_points) / sfreq
    n_channels = len(channel_names)
    fig, axes = plt.subplots(n_channels, 2, figsize=(14, 1.1 * n_channels), sharex=True)
    for i in range(n_channels):
        axes[i, 0].plot(t, X_before[idx, i, :n_points], linewidth=0.6, color='tab:blue')
        axes[i, 1].plot(t, X_after[idx, i, :n_points], linewidth=0.6, color='tab:green')
        axes[i, 0].set_ylabel(channel_names[i], rotation=0, ha='right', fontsize=7)
        axes[i, 0].set_yticks([]); axes[i, 1].set_yticks([])
    axes[0, 0].set_title('Before ICA cleaning')
    axes[0, 1].set_title('After ICA cleaning (v2)')
    axes[-1, 0].set_xlabel('Time (s)'); axes[-1, 1].set_xlabel('Time (s)')
    plt.tight_layout()
    plt.savefig(f'{PROJECT_DIR}/figures/ica_before_after_v2.png', dpi=150)
    plt.show()

plot_before_after_ica(X_filtered, X_clean, idx=0)

**Check the plot above against last run's version.** The frontal spikes should now be visibly smaller than before. If a handful of individual subjects still show little improvement (check the "no component flagged" list printed above), that's plausible -- `find_bads_eog`'s automatic threshold won't catch every subject perfectly, especially in a proxy-channel setup like this without true EOG. Perfect cleaning on all 45 subjects isn't the bar; a clear, visible improvement in the aggregate is.

## 3. Windowing

**What this does:** cuts each 150-second recording into shorter, overlapping windows so we have enough labeled examples to train on, while preserving the link back to each subject for later grouping.

**Why 4-second windows with 50% overlap:** short enough to give hundreds of examples, long enough to contain multiple cycles of even the slowest band we care about (theta, ~4Hz).

In [ ]:
WINDOW_SECONDS = 4
OVERLAP = 0.5
window_size = int(WINDOW_SECONDS * SFREQ)
step_size = int(window_size * (1 - OVERLAP))

def window_data(X, y, subject_id, window_size, step_size):
    X_windows, y_windows, subj_windows = [], [], []
    n_timepoints = X.shape[2]
    for i in range(X.shape[0]):
        start = 0
        while start + window_size <= n_timepoints:
            X_windows.append(X[i, :, start:start + window_size])
            y_windows.append(y[i])
            subj_windows.append(subject_id[i])
            start += step_size
    return np.array(X_windows), np.array(y_windows), np.array(subj_windows)

X_windows, y_windows, subj_windows = window_data(X_clean, y, subject_id, window_size, step_size)

print('X_windows shape:', X_windows.shape)
print('y_windows shape:', y_windows.shape)
print('Windows per subject (should be a single repeated value):',
      pd.Series(subj_windows).value_counts().unique())
print('Total windows:', len(y_windows), '| Class balance:', pd.Series(y_windows).value_counts(normalize=True).sort_index().to_dict())

## 4. Band-power feature extraction

**What this does:** for each window, for each channel, compute average power in theta (4-8Hz), alpha (8-13Hz), beta (13-30Hz), and gamma (30-45Hz) bands -- collapsing each window from 14 x 512 raw numbers into 14 x 4 = 56 theory-driven features.

In [ ]:
from scipy.integrate import trapezoid

BANDS = {'theta': (4, 8), 'alpha': (8, 13), 'beta': (13, 30), 'gamma': (30, 45)}

def compute_band_power(window, sfreq=SFREQ, bands=BANDS):
    n_channels = window.shape[0]
    band_powers = {band: np.zeros(n_channels) for band in bands}
    for ch in range(n_channels):
        freqs, psd = welch(window[ch], fs=sfreq, nperseg=min(256, window.shape[1]))
        for band, (lo, hi) in bands.items():
            mask = (freqs >= lo) & (freqs <= hi)
            band_powers[band][ch] = trapezoid(psd[mask], freqs[mask])
    return band_powers

def extract_features(X_windows, sfreq=SFREQ, bands=BANDS, channel_names=CHANNEL_NAMES):
    feature_rows = []
    feature_names = [f'{band}_{ch}' for band in bands for ch in channel_names]
    for w in range(X_windows.shape[0]):
        band_powers = compute_band_power(X_windows[w], sfreq=sfreq, bands=bands)
        row = np.concatenate([band_powers[band] for band in bands])
        feature_rows.append(row)
    return np.array(feature_rows), feature_names

X_features, feature_names = extract_features(X_windows)
print('Feature matrix shape:', X_features.shape, '(n_windows, n_features)')
print('First few feature names:', feature_names[:6], '...')

## 4.5. Per-subject normalization -- tried, and here's why it doesn't fit this dataset

**The idea (kept here for transparency):** normalize each subject's band power against their own mean/std, to remove between-person anatomical variance (skull thickness, electrode contact, etc.) that has nothing to do with cognitive state -- a standard technique in EEG research.

**Why it doesn't apply here, discovered empirically:** in STEW, each subject contributes exactly one recording and one label -- every one of a subject's windows shares the identical workload class. That means any real classification signal *must* live in differences *between* subjects, since there's no within-subject contrast (no paired low-vs-high recording from the same person) to compare against. Per-subject z-scoring forces every subject's own mean to exactly 0 regardless of their label -- which removes exactly the between-subject signal the task depends on, not an anatomical confound.

**This was confirmed by running it:** the Section 5 theory-check medians all collapsed to nearly identical values across classes, and subject-independent model accuracy dropped to/below chance (RF 0.289, SVM 0.337) -- both symptoms of removing the actual target signal, not evidence of a cleaner analysis. Per-subject normalization is the right tool for studies with repeated within-subject conditions; it's the wrong tool for a one-label-per-subject design like this one. We're keeping this cell as a documented negative result rather than deleting it -- it's a legitimate, explainable methodological finding worth mentioning in a write-up, not a mistake to hide.

**Going forward:** Sections 5-7 below use the raw (non-per-subject-normalized) band-power features from Section 4, which is what produced our earlier, more sensible results.

In [ ]:
# Per-subject normalization was tested here and found to remove the target
# signal itself (see markdown above) -- so we deliberately do NOT apply it.
# X_features (raw, from Section 4) is used directly in Sections 5-7 below.
print('Using raw (non-per-subject-normalized) features for all downstream analysis.')
print('X_features shape:', X_features.shape)

## 5. Sanity-check features against cognitive load theory (median-based)

**What this does:** checks whether frontal theta power trends *up* and posterior alpha power trends *down* as workload increases, per cognitive load theory, using the raw band-power features. We report medians (robust to residual-artifact-driven outliers) alongside means, and hide outlier points in the boxplots so the typical-window distribution is actually visible.

In [ ]:
df_features = pd.DataFrame(X_features, columns=feature_names)
df_features['label'] = y_windows

FRONTAL_CHANNELS = ['AF3', 'F7', 'F8', 'AF4']
POSTERIOR_CHANNELS = ['O1', 'O2', 'P7', 'P8']
frontal_theta_cols = [f'theta_{ch}' for ch in FRONTAL_CHANNELS]
posterior_alpha_cols = [f'alpha_{ch}' for ch in POSTERIOR_CHANNELS]

df_features['frontal_theta_avg'] = df_features[frontal_theta_cols].mean(axis=1)
df_features['posterior_alpha_avg'] = df_features[posterior_alpha_cols].mean(axis=1)

summary_median = df_features.groupby('label')[['frontal_theta_avg', 'posterior_alpha_avg']].median()
summary_median.index = ['low (0)', 'moderate (1)', 'high (2)']
print('--- Median (robust to outliers) ---')
print(summary_median)

summary_mean = df_features.groupby('label')[['frontal_theta_avg', 'posterior_alpha_avg']].mean()
summary_mean.index = ['low (0)', 'moderate (1)', 'high (2)']
print('\n--- Mean (for comparison -- expect this to look noisier/outlier-skewed) ---')
print(summary_mean)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.boxplot(data=df_features, x='label', y='frontal_theta_avg', ax=axes[0], showfliers=False)
axes[0].set_title('Frontal theta power by workload class')
axes[0].set_xticks([0, 1, 2])
axes[0].set_xticklabels(['low', 'moderate', 'high'])
sns.boxplot(data=df_features, x='label', y='posterior_alpha_avg', ax=axes[1], showfliers=False)
axes[1].set_title('Posterior alpha power by workload class')
axes[1].set_xticks([0, 1, 2])
axes[1].set_xticklabels(['low', 'moderate', 'high'])
plt.tight_layout()
plt.savefig(f'{PROJECT_DIR}/figures/theory_check_boxplots_v3.png', dpi=150)
plt.show()

**How to read this.** Compare the median lines and overall spread of the boxes across classes. A partial or noisy trend is normal for real data at n=45; last two runs showed a fairly flat pattern even before/after ICA changes, which may simply reflect that a 2-channel-average summary is too coarse to reveal a subtle effect at this sample size -- the full 56-feature model in Section 6 is a separate, still-valid test that doesn't depend on this simpler summary showing a clean trend.

## 6. Baseline models with subject-independent cross-validation

**What this does:** trains Random Forest and SVM classifiers using `GroupKFold` grouped by `subj_windows` -- every fold's test set contains only subjects never seen during training on that fold.

**Why this is the right way to evaluate:** windows from the same subject share subject-specific signal (skull thickness, baseline brain rhythm, electrode fit) unrelated to workload. A model can exploit that to "recognize" a subject rather than learn workload patterns.

In [ ]:
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

N_SPLITS = 5
gkf = GroupKFold(n_splits=N_SPLITS)

models = {
    'RandomForest': RandomForestClassifier(n_estimators=200, random_state=RANDOM_SEED, class_weight='balanced'),
    'SVM': Pipeline([('scaler', StandardScaler()), ('svm', SVC(kernel='rbf', class_weight='balanced'))]),
}

results_subject_independent = {}
for name, model in models.items():
    scores = cross_val_score(model, X_features, y_windows, cv=gkf, groups=subj_windows, scoring='accuracy')
    results_subject_independent[name] = scores
    print(f'{name}: fold accuracies = {np.round(scores, 3)}, mean = {scores.mean():.3f} +/- {scores.std():.3f}')

chance_level = 1 / len(np.unique(y_windows))
print(f'\nChance level (3-class): {chance_level:.3f}')

**How to read this.** Compare mean accuracy to chance level (~0.333). Meaningfully above chance is a legitimate, defensible result for a small 3-class EEG workload task. Last run's numbers (RF 0.425, SVM 0.483) were already reasonable -- with cleaner ICA this time, watch whether these numbers hold steady, improve slightly, or stay about the same. A small change either way is fine; don't expect a dramatic jump from this fix alone, since the outlier contamination mainly affected the theory-check averages, not necessarily the trained models (which use the full feature vector, not just the two summary channels we averaged in Section 5).

## 7. Demonstrating the leakage problem directly

**What this does:** repeats the same modeling process with a naive random K-fold split that ignores subject grouping -- windows from the same subject can land in both train and test. Deliberately the *wrong* way to evaluate, done on purpose to quantify the gap.

**What we expect to see:** noticeably higher (inflated) accuracy than the subject-independent version -- as we already saw dramatically with Random Forest (0.425 -> 0.971) last run.

In [ ]:
from sklearn.model_selection import KFold

kf_naive = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_SEED)

results_naive = {}
for name, model in models.items():
    scores = cross_val_score(model, X_features, y_windows, cv=kf_naive, scoring='accuracy')
    results_naive[name] = scores
    print(f'{name} (naive split): mean = {scores.mean():.3f} +/- {scores.std():.3f}')

print('\n--- Comparison ---')
for name in models:
    si = results_subject_independent[name].mean()
    naive = results_naive[name].mean()
    print(f'{name}: subject-independent = {si:.3f} | naive (leaky) = {naive:.3f} | inflation = {naive - si:+.3f}')

**This comparison is itself a result worth keeping in your write-up.** Last run's finding -- RF inflated by +0.546 while SVM barely moved (-0.008) -- is a genuinely interesting, explainable contrast: tree-based models can carve very specific decision boundaries that memorize subject-specific quirks, while SVM's smoother margin is less prone to that. If this pattern holds again with cleaner data, it's a strong, specific talking point.

## 8. Checkpoint: save features and results

In [ ]:
np.save(f'{PROJECT_DIR}/data/processed/X_features.npy', X_features)
np.save(f'{PROJECT_DIR}/data/processed/y_windows.npy', y_windows)
np.save(f'{PROJECT_DIR}/data/processed/subj_windows.npy', subj_windows)
with open(f'{PROJECT_DIR}/data/processed/feature_names.txt', 'w') as f:
    f.write('\n'.join(feature_names))

print('Checkpoint saved.')

## Next steps (Notebook 3)

1. Try a 1D-CNN or LSTM directly on filtered raw windows (not hand-engineered features) and compare to this baseline
2. SHAP explainability on the Random Forest / SVM baseline -- which bands/channels actually drove predictions
3. Decide whether to keep 3-class or collapse to binary for the deployed demo
4. Build the Streamlit/Gradio deployment app